# Scikit-learn 机器学习算法实战

本 notebook 系统覆盖 sklearn 常用算法，每个算法包含：原理简述 → 代码实现 → 评估指标 → 可视化。

**环境**：`conda activate main`

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split, cross_val_score
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix, \
    mean_squared_error, r2_score, silhouette_score

plt.rcParams['font.sans-serif'] = ['SimHei', 'Microsoft YaHei']
plt.rcParams['axes.unicode_minus'] = False

np.random.seed(42)
print("所有库导入成功")

---
## 0. 数据集准备

In [ ]:
from sklearn.datasets import make_classification, make_regression, make_blobs

# 分类数据集
X_cls, y_cls = make_classification(
    n_samples=500, n_features=10, n_informative=5, n_redundant=2,
    n_classes=3, random_state=42
)
X_train_cls, X_test_cls, y_train_cls, y_test_cls = train_test_split(
    X_cls, y_cls, test_size=0.2, random_state=42
)

# 回归数据集
X_reg, y_reg = make_regression(n_samples=500, n_features=5, noise=10, random_state=42)
X_train_reg, X_test_reg, y_train_reg, y_test_reg = train_test_split(
    X_reg, y_reg, test_size=0.2, random_state=42
)

# 聚类数据集
X_clust, y_clust = make_blobs(n_samples=300, centers=4, cluster_std=1.0, random_state=42)

print(f"分类: train={X_train_cls.shape}, test={X_test_cls.shape}")
print(f"回归: train={X_train_reg.shape}, test={X_test_reg.shape}")
print(f"聚类: {X_clust.shape}")

---
## 1. 线性回归（Linear Regression）

**原理**：寻找最佳直线 y = wx + b，使预测值与真实值的平方误差最小

**适用场景**：房价预测、销售额预测等连续值预测

In [ ]:
from sklearn.linear_model import LinearRegression

model = LinearRegression()
model.fit(X_train_reg, y_train_reg)

y_pred = model.predict(X_test_reg)
print(f"MSE:  {mean_squared_error(y_test_reg, y_pred):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred)):.2f}")
print(f"R²:   {r2_score(y_test_reg, y_pred):.4f}")
print(f"\n特征系数: {model.coef_.round(2)}")
print(f"截距: {model.intercept_:.2f}")

In [ ]:
# 可视化：真实值 vs 预测值
fig, ax = plt.subplots(figsize=(6, 6))
ax.scatter(y_test_reg, y_pred, alpha=0.5)
ax.plot([y_test_reg.min(), y_test_reg.max()],
        [y_test_reg.min(), y_test_reg.max()], 'r--', lw=2)
ax.set_xlabel('真实值')
ax.set_ylabel('预测值')
ax.set_title(f'线性回归 R²={r2_score(y_test_reg, y_pred):.4f}')
plt.tight_layout()
plt.show()

---
## 2. 逻辑回归（Logistic Regression）

**原理**：在线性回归基础上加 Sigmoid 函数，将输出映射到 [0,1] 概率值

**适用场景**：二分类/多分类（垃圾邮件、疾病诊断）

In [ ]:
from sklearn.linear_model import LogisticRegression

# 二分类
X_binary, y_binary = make_classification(n_samples=500, n_features=10,
                                         n_classes=2, random_state=42)
X_tr, X_te, y_tr, y_te = train_test_split(X_binary, y_binary, test_size=0.2, random_state=42)

model = LogisticRegression(max_iter=1000)
model.fit(X_tr, y_tr)

y_pred = model.predict(X_te)
y_prob = model.predict_proba(X_te)[:, 1]

print(f"准确率: {accuracy_score(y_te, y_pred):.4f}")
print(f"\n分类报告:\n{classification_report(y_te, y_pred)}")

In [ ]:
# 混淆矩阵
fig, ax = plt.subplots(figsize=(5, 4))
cm = confusion_matrix(y_te, y_pred)
sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', ax=ax)
ax.set_xlabel('预测值')
ax.set_ylabel('真实值')
ax.set_title('逻辑回归混淆矩阵')
plt.tight_layout()
plt.show()

---
## 3. K近邻（KNN）

**原理**：找到距离最近的 K 个邻居，投票决定分类

**关键**：K值选择、距离度量、特征标准化

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

# KNN 对特征尺度敏感，需要标准化
scaler = StandardScaler()
X_tr_scaled = scaler.fit_transform(X_train_cls)
X_te_scaled = scaler.transform(X_test_cls)

# 不同K值的效果
k_range = range(1, 21)
scores = []
for k in k_range:
    knn = KNeighborsClassifier(n_neighbors=k)
    knn.fit(X_tr_scaled, y_train_cls)
    scores.append(knn.score(X_te_scaled, y_test_cls))

best_k = list(k_range)[np.argmax(scores)]
print(f"最佳K值: {best_k}, 准确率: {max(scores):.4f}")

# 可视化
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(k_range, scores, 'bo-')
ax.axvline(x=best_k, color='red', linestyle='--', label=f'best K={best_k}')
ax.set_xlabel('K值')
ax.set_ylabel('准确率')
ax.set_title('KNN: K值选择')
ax.legend()
plt.tight_layout()
plt.show()

---
## 4. 决策树（Decision Tree）

**原理**：通过一系列 if-else 规则将数据递归划分，形成树结构

**优点**：可解释性强  |  **缺点**：容易过拟合

In [ ]:
from sklearn.tree import DecisionTreeClassifier, plot_tree

dt = DecisionTreeClassifier(max_depth=4, random_state=42)
dt.fit(X_train_cls, y_train_cls)

y_pred = dt.predict(X_test_cls)
print(f"准确率: {accuracy_score(y_test_cls, y_pred):.4f}")
print(f"训练集准确率: {dt.score(X_train_cls, y_train_cls):.4f}")

In [ ]:
# 可视化决策树
fig, ax = plt.subplots(figsize=(16, 8))
plot_tree(dt, max_depth=3, filled=True, rounded=True,
          feature_names=[f'F{i}' for i in range(10)],
          class_names=[f'类{i}' for i in range(3)],
          ax=ax, fontsize=9)
ax.set_title('决策树可视化（深度限制=4）')
plt.tight_layout()
plt.show()

In [ ]:
# 特征重要性
importance = dt.feature_importances_
indices = np.argsort(importance)[::-1]

fig, ax = plt.subplots(figsize=(8, 4))
ax.bar(range(10), importance[indices])
ax.set_xticks(range(10))
ax.set_xticklabels([f'F{i}' for i in indices])
ax.set_title('决策树特征重要性')
plt.tight_layout()
plt.show()

---
## 5. 随机森林（Random Forest）

**原理**：多棵决策树的集成（bagging），每棵树用随机子集训练，最终投票

**优势**：比单棵决策树更稳定，不容易过拟合

In [ ]:
from sklearn.ensemble import RandomForestClassifier

rf = RandomForestClassifier(n_estimators=100, max_depth=8, random_state=42)
rf.fit(X_train_cls, y_train_cls)

y_pred = rf.predict(X_test_cls)
print(f"随机森林准确率: {accuracy_score(y_test_cls, y_pred):.4f}")
print(f"决策树准确率:   {accuracy_score(y_test_cls, dt.predict(X_test_cls)):.4f}")

# 交叉验证
cv_scores = cross_val_score(rf, X_cls, y_cls, cv=5, scoring='accuracy')
print(f"\n5折交叉验证: {cv_scores.round(4)}")
print(f"平均准确率: {cv_scores.mean():.4f} (+/- {cv_scores.std():.4f})")

---
## 6. 支持向量机（SVM）

**原理**：找到一个超平面，使不同类别之间的间隔（margin）最大化

**核心**：核函数（kernel）将数据映射到高维空间解决非线性问题

In [ ]:
from sklearn.svm import SVC

# SVM 对特征尺度敏感
scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train_cls)
X_te_s = scaler.transform(X_test_cls)

# 不同核函数对比
kernels = ['linear', 'rbf', 'poly']
for kernel in kernels:
    svm = SVC(kernel=kernel, random_state=42)
    svm.fit(X_tr_s, y_train_cls)
    acc = svm.score(X_te_s, y_test_cls)
    print(f"{kernel:>8} 核: 准确率 = {acc:.4f}")

---
## 7. 梯度提升（Gradient Boosting）

**原理**：顺序训练弱学习器，每棵树修正前一棵的错误（boosting）

**应用**：竞赛中最常用的算法之一

In [ ]:
from sklearn.ensemble import GradientBoostingClassifier

gb = GradientBoostingClassifier(
    n_estimators=100, learning_rate=0.1, max_depth=3, random_state=42
)
gb.fit(X_train_cls, y_train_cls)

y_pred = gb.predict(X_test_cls)
print(f"GradientBoosting 准确率: {accuracy_score(y_test_cls, y_pred):.4f}")

# 查看训练过程
fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(range(1, 101), gb.train_score_, 'b-')
ax.set_xlabel('迭代次数')
ax.set_ylabel('训练损失')
ax.set_title('GradientBoosting 训练曲线')
plt.tight_layout()
plt.show()

---
## 8. K-Means 聚类

**原理**：无监督算法，将数据分成 K 个簇，使簇内距离最小、簇间距离最大

In [ ]:
from sklearn.cluster import KMeans

# 肘部法则选K
inertias = []
sil_scores = []
K_range = range(2, 10)
for k in K_range:
    km = KMeans(n_clusters=k, random_state=42, n_init=10)
    km.fit(X_clust)
    inertias.append(km.inertia_)
    sil_scores.append(silhouette_score(X_clust, km.labels_))

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(K_range, inertias, 'bo-')
axes[0].set_xlabel('K')
axes[0].set_ylabel('Inertia')
axes[0].set_title('肘部法则')

axes[1].plot(K_range, sil_scores, 'ro-')
axes[1].set_xlabel('K')
axes[1].set_ylabel('轮廓系数')
axes[1].set_title('轮廓系数法')
plt.tight_layout()
plt.show()

In [ ]:
# 最终聚类
best_k = 4
km = KMeans(n_clusters=best_k, random_state=42, n_init=10)
labels = km.fit_predict(X_clust)

fig, ax = plt.subplots(figsize=(7, 6))
scatter = ax.scatter(X_clust[:, 0], X_clust[:, 1], c=labels, cmap='viridis', alpha=0.7)
ax.scatter(km.cluster_centers_[:, 0], km.cluster_centers_[:, 1],
           c='red', marker='X', s=200, edgecolors='black', label='聚类中心')
ax.set_title(f'K-Means 聚类 (K={best_k})')
ax.legend()
plt.colorbar(scatter)
plt.tight_layout()
plt.show()

---
## 9. 模型评估全览

### 分类指标

In [ ]:
from sklearn.metrics import precision_score, recall_score, f1_score, roc_auc_score

# 多分类逻辑回归
lr = LogisticRegression(max_iter=1000)
lr.fit(X_train_cls, y_train_cls)
y_pred = lr.predict(X_test_cls)
y_prob = lr.predict_proba(X_test_cls)

print("=== 分类指标 ===")
print(f"准确率 (Accuracy):  {accuracy_score(y_test_cls, y_pred):.4f}")
print(f"精确率 (Precision): {precision_score(y_test_cls, y_pred, average='macro'):.4f}")
print(f"召回率 (Recall):    {recall_score(y_test_cls, y_pred, average='macro'):.4f}")
print(f"F1 分数:            {f1_score(y_test_cls, y_pred, average='macro'):.4f}")

# 交叉验证
cv = cross_val_score(lr, X_cls, y_cls, cv=5)
print(f"\n5折CV: {cv.mean():.4f} (+/- {cv.std():.4f})")

### 回归指标

In [ ]:
lr_reg = LinearRegression()
lr_reg.fit(X_train_reg, y_train_reg)
y_pred_r = lr_reg.predict(X_test_reg)

print("=== 回归指标 ===")
print(f"MSE:  {mean_squared_error(y_test_reg, y_pred_r):.2f}")
print(f"RMSE: {np.sqrt(mean_squared_error(y_test_reg, y_pred_r)):.2f}")
print(f"MAE:  {np.mean(np.abs(y_test_reg - y_pred_r)):.2f}")
print(f"R²:   {r2_score(y_test_reg, y_pred_r):.4f}")

---
## 10. 算法对比总览

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.svm import SVC
from sklearn.naive_bayes import GaussianNB

models = {
    '逻辑回归': LogisticRegression(max_iter=1000),
    'KNN': KNeighborsClassifier(n_neighbors=5),
    '决策树': DecisionTreeClassifier(max_depth=8, random_state=42),
    '随机森林': RandomForestClassifier(n_estimators=100, random_state=42),
    'GradientBoosting': GradientBoostingClassifier(n_estimators=100, random_state=42),
    'SVM': SVC(random_state=42),
    '朴素贝叶斯': GaussianNB()
}

scaler = StandardScaler()
X_tr_s = scaler.fit_transform(X_train_cls)
X_te_s = scaler.transform(X_test_cls)

results = []
for name, model in models.items():
    # SVM 和 KNN 用标准化数据
    if name in ['SVM', 'KNN']:
        model.fit(X_tr_s, y_train_cls)
        acc = model.score(X_te_s, y_test_cls)
    else:
        model.fit(X_train_cls, y_train_cls)
        acc = model.score(X_test_cls, y_test_cls)
    results.append({'模型': name, '准确率': acc})

df_results = pd.DataFrame(results).sort_values('准确率', ascending=False)
print(df_results.to_string(index=False))

# 可视化对比
fig, ax = plt.subplots(figsize=(8, 5))
colors = plt.cm.RdYlGn(np.linspace(0.3, 0.9, len(df_results)))
ax.barh(df_results['模型'], df_results['准确率'], color=colors)
ax.set_xlabel('准确率')
ax.set_title('分类算法准确率对比')
for i, (name, acc) in enumerate(zip(df_results['模型'], df_results['准确率'])):
    ax.text(acc + 0.002, i, f'{acc:.4f}', va='center')
ax.set_xlim(0, 1)
plt.tight_layout()
plt.show()

---
## 算法选择指南

| 场景 | 推荐算法 | 原因 |
|------|----------|------|
| 数据量小 + 特征少 | KNN、逻辑回归 | 简单高效 |
| 数据量大 + 需要解释 | 决策树、随机森林 | 可解释性好 |
| 追求高精度 | GradientBoosting、随机森林 | 集成方法鲁棒性强 |
| 线性可分 | 逻辑回归、SVM(linear) | 快速收敛 |
| 非线性问题 | SVM(rbf)、随机森林 | 核方法/集成 |
| 无标签数据 | K-Means、层次聚类 | 无监督 |

**核心流程**：数据探索 → 特征工程 → 模型选择 → 训练调参 → 评估验证

**下一步**：学习模型调参（网格搜索、交叉验证） → 数据预处理实战